In [4]:
import pandas as pd

fake = pd.read_csv('../data/Fake.csv')
real = pd.read_csv('../data/True.csv')

print(fake.shape, real.shape)
fake.head()

(23481, 4) (21417, 4)


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
print("Fake subjects:", fake['subject'].unique())
print("Real subjects:", real['subject'].unique())


Fake subjects: <StringArray>
['News', 'politics', 'Government News', 'left-news', 'US_News', 'Middle-east']
Length: 6, dtype: str
Real subjects: <StringArray>
['politicsNews', 'worldnews']
Length: 2, dtype: str


In [6]:
print(fake['subject'].value_counts())

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64


In [7]:
fake['label'] = 'Fake News'
real['label'] = 'Real News'

data = pd.concat([fake[['title', 'text', 'label']], real[['title', 'text', 'label']]], ignore_index=True)

print(data.shape)
print(data['label'].value_counts())
data.sample(5)

(44898, 3)
label
Fake News    23481
Real News    21417
Name: count, dtype: int64


,title,text,label
35951,France's Le Drian says 'no' to Iran Mediterran...,PARIS (Reuters) - France s foreign minister cr...,Real News
29859,Mattis to say U.S. must confront Russian behav...,WASHINGTON (Reuters) - President-elect Donald ...,Real News
11080,CLASSIC TRUMP! President Trump Announces Where...,On the eve of the upcoming 2017 White House Co...,Fake News
17693,LOL! MOTHER OF TWO Just Got A BIG SURPRISE Fro...,"The picture, snapped by a White House photogra...",Fake News
29564,Microsoft victory in overseas email seizure ca...,NEW YORK (Reuters) - An equally divided federa...,Real News


In [8]:
reuters_count = data[data['label'] == 'Real News']['text'].str.contains('Reuters', case=False).sum()
total_real = (data['label'] == 'Real News').sum()
print(f"{reuters_count} out of {total_real} real articles mention 'Reuters'")

twitter_count = data[data['label'] == 'Fake News']['text'].str.contains('Reuters', case=False).sum()
total_fake = (data['label'] == 'Fake News').sum()
print(f"{twitter_count} out of {total_fake} fake articles mention 'Reuters'")

21378 out of 21417 real articles mention 'Reuters'
322 out of 23481 fake articles mention 'Reuters'


In [9]:
for t in data[data['label'] == 'Real News']['text'].head(5):
    print(t[:100])
    print('---')

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted
---
WASHINGTON (Reuters) - Transgender people will be allowed for the first time to enlist in the U.S. m
---
WASHINGTON (Reuters) - The special counsel investigation of links between Russia and President Trump
---
WASHINGTON (Reuters) - Trump campaign adviser George Papadopoulos told an Australian diplomat in May
---
SEATTLE/WASHINGTON (Reuters) - President Donald Trump called on the U.S. Postal Service on Friday to
---


In [11]:
import re

def remove_reuters_dateline(text):
    match = re.match(r'^.{0,100}?\(Reuters\)\s*-\s*', text)
    if match:
        return text[match.end():]
    return text

data['text'] = data['text'].apply(remove_reuters_dateline)

reuters_count = data[data['label'] == 'Real News']['text'].str.contains('Reuters', case=False).sum()
print(f"{reuters_count} real articles still mention 'Reuters' after cleanup")

5030 real articles still mention 'Reuters' after cleanup


In [14]:
reuters_count = data[data['label'] == 'Real News']['text'].str.contains('Reuters', case=False).sum()
print(f"{reuters_count} real articles still mention 'Reuters' after cleanup")

5029 real articles still mention 'Reuters' after cleanup


In [12]:
sample_with_reuters = data[(data['label'] == 'Real News') & (data['text'].str.contains('Reuters', case=False))]['text'].head(5)
for t in sample_with_reuters:
    idx = t.lower().find('reuters')
    print(t[max(0, idx-50):idx+50])
    print('---')

poulos did not immediately respond to requests by Reuters for comment. Mueller’s office declined to 
---
 and @POTUS.  The opinions expressed are his own. Reuters has not edited the statements or confirmed
---
 and @POTUS.  The opinions expressed are his own. Reuters has not edited the statements or confirmed
---
n his early 30s.  Moore has denied wrongdoing and Reuters has not been able to independently verify 
---
t Indiana University’s Maurer School of Law, told Reuters last week, speaking generally about govern
---


In [13]:
real_with_reuters = data[(data['label'] == 'Real News') & (data['text'].str.contains('Reuters', case=False))]
positions = real_with_reuters['text'].apply(lambda t: t.lower().find('reuters'))
print(positions.describe())

count     5030.000000
mean      1363.957654
std       1288.554579
min          0.000000
25%        398.000000
50%        957.000000
75%       1967.750000
max      21574.000000
Name: text, dtype: float64


In [14]:
near_start = (positions < 20).sum()
print(f"{near_start} articles have 'Reuters' within the first 20 characters")

21 articles have 'Reuters' within the first 20 characters


In [18]:
stop_words = set(stopwords.words('english'))
stop_words.add('reuters')

NameError: name 'stopwords' is not defined

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
stop_words.add('reuters')
stop_words.add('via')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ekans\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ekans\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [16]:
data['text_length'] = data['text'].apply(lambda t: len(t.split()))
print(data['text_length'].describe())
print(data.groupby('label')['text_length'].describe())

count    44898.000000
mean       403.794156
std        351.329647
min          0.000000
25%        202.000000
50%        361.000000
75%        512.000000
max       8135.000000
Name: text_length, dtype: float64
             count        mean         std  min    25%    50%    75%     max
label                                                                       
Fake News  23481.0  423.197905  408.388890  0.0  240.0  363.0  506.0  8135.0
Real News  21417.0  382.520428  273.945422  0.0  146.0  356.0  522.0  5170.0


In [21]:
empty_articles = data[data['text_length'] == 0]
print(f"{len(empty_articles)} empty articles")
print(empty_articles['label'].value_counts())
empty_articles[['title', 'text']].head()

631 empty articles
label
Fake News    630
Real News      1
Name: count, dtype: int64


,title,text
10923,TAKE OUR POLL: Who Do You Think President Trum...,
11041,Joe Scarborough BERATES Mika Brzezinski Over “...,
11190,WATCH TUCKER CARLSON Scorch Sanctuary City May...,
11225,MAYOR OF SANCTUARY CITY: Trump Trying To Make ...,
11236,SHOCKER: Public School Turns Computer Lab Into...,


In [22]:
data = data[data['text_length'] > 0]
print(data.shape)
print(data['label'].value_counts())

(44267, 4)
label
Fake News    22851
Real News    21416
Name: count, dtype: int64


In [25]:
import re
import contractions

negation_words = {'not', 'no', 'nor', 'never', "didn't", "don't", "won't", "isn't", "wasn't", "aren't"}
stop_words = stop_words - negation_words

def clean_text(text):
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)      # remove any URLs
    text = re.sub(r'[^a-z\s]', ' ', text)            # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

data['text_clean'] = data['text'].apply(clean_text)
data[['text', 'text_clean']].head(3)

,text,text_clean
0,Donald Trump just couldn t wish all Americans ...,donald trump wish american happy new year leav...
1,House Intelligence Committee Chairman Devin Nu...,house intelligence committee chairman devin nu...
2,"On Friday, it was revealed that former Milwauk...",friday revealed former milwaukee sheriff david...


In [24]:
pip install contractions


  Using cached contractions-0.1.73-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached textsearch-0.0.24-py2.py3-none-any.whl.metadata (1.2 kB)
  Using cached anyascii-0.3.3-py3-none-any.whl.metadata (1.6 kB)
  Using cached pyahocorasick-2.3.1-cp312-cp312-win_amd64.whl.metadata (14 kB)
Using cached contractions-0.1.73-py2.py3-none-any.whl (8.7 kB)
Using cached textsearch-0.0.24-py2.py3-none-any.whl (7.6 kB)
Using cached anyascii-0.3.3-py3-none-any.whl (345 kB)
Using cached pyahocorasick-2.3.1-cp312-cp312-win_amd64.whl (35 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
broken_contractions = data['text'].str.contains(r"\b(couldn|didn|wasn|isn|wouldn|shouldn|aren|weren) t\b", regex=True, case=False).sum()
print(f"{broken_contractions} articles have space-broken contractions")

C:\Users\ekans\AppData\Local\Temp\ipykernel_9544\655329023.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  broken_contractions = data['text'].str.contains(r"\b(couldn|didn|wasn|isn|wouldn|shouldn|aren|weren) t\b", regex=True, case=False).sum()


10201 articles have space-broken contractions


In [27]:
def repair_broken_contractions(text):
    # Turn "couldn t" -> "couldn't", "didn t" -> "didn't", etc.
    text = re.sub(r"\b(couldn|didn|wasn|isn|wouldn|shouldn|aren|weren|hasn|haven|hadn|doesn|don) t\b", r"\1't", text, flags=re.IGNORECASE)
    return text

In [28]:
def clean_text(text):
    text = repair_broken_contractions(text)
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

data['text_clean'] = data['text'].apply(clean_text)
data[['text', 'text_clean']].head(3)

,text,text_clean
0,Donald Trump just couldn t wish all Americans ...,donald trump could not wish american happy new...
1,House Intelligence Committee Chairman Devin Nu...,house intelligence committee chairman devin nu...
2,"On Friday, it was revealed that former Milwauk...",friday revealed former milwaukee sheriff david...


In [29]:
empty_count = (data['text_clean'].str.strip() == '').sum()
print(f"Empty cleaned articles: {empty_count}")


Empty cleaned articles: 85


In [30]:
data = data[data['text_clean'].str.strip() != '']
print(data.shape)

(44182, 5)


In [27]:
import pandas as pd
import re
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
negation_words = {'not', 'no', 'nor', 'never', "didn't", "don't", "won't", "isn't", "wasn't", "aren't"}
stop_words = stop_words - negation_words
stop_words.add('reuters')
stop_words.add('via')
stop_words.add('getty')
stop_words.add('screenshot')

lemmatizer = WordNetLemmatizer()

# Reload raw data fresh
fake = pd.read_csv('../data/Fake.csv')
real = pd.read_csv('../data/True.csv')

fake['label'] = 'Fake News'
real['label'] = 'Real News'

data = pd.concat([fake[['title', 'text', 'label']], real[['title', 'text', 'label']]], ignore_index=True)

def remove_reuters_dateline(text):
    match = re.match(r'^.{0,100}?\(Reuters\)\s*-\s*', text)
    if match:
        return text[match.end():]
    return text

def remove_media_credits(text):
    text = re.sub(r'featured image[:\s](by|from)?.{0,60}', '', text, flags=re.IGNORECASE)
    return text

def repair_broken_contractions(text):
    text = re.sub(r"\b(couldn|didn|wasn|isn|wouldn|shouldn|aren|weren|hasn|haven|hadn|doesn|don) t\b", r"\1't", text, flags=re.IGNORECASE)
    return text

def clean_text(text):
    text = remove_reuters_dateline(text)
    text = remove_media_credits(text)
    text = repair_broken_contractions(text)
    text = text.lower()
    text = contractions.fix(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

data['text_clean'] = data['text'].apply(clean_text)
data['text_length'] = data['text_clean'].apply(lambda t: len(t.split()))
data = data[data['text_length'] > 0]

print(data.shape)
print(data['label'].value_counts())
data[['text', 'text_clean']].sample(5)

(44182, 5)
label
Fake News    22766
Real News    21416
Name: count, dtype: int64


,text,text_clean
4569,"A Dallas police officer, seeing red after seve...",dallas police officer seeing red several shoot...
24563,(Reuters) - Iowa on Monday withdrew a request ...,iowa monday withdrew request waive obamacare r...
32313,WASHINGTON (Reuters) - The U.S. State Departme...,state department said thursday conduct interna...
42710,DAKAR/ACCRA (Reuters) - The International Trib...,international tribunal law sea saturday drew o...
10801,WOW! THIS IS FANTASTIC! WE HIGHLY RECOMMEND TH...,wow fantastic highly recommend entire video of...


In [28]:
import re

def remove_image_credits(text):
    # Removes "Featured image via ... " and similar credit-line patterns, wherever they appear
    text = re.sub(r'featured image (via|credit from).{0,60}', '', text, flags=re.IGNORECASE)
    return text

data['text'] = data['text'].apply(remove_image_credits)

featured_image_count = data[data['label'] == 'Fake News']['text'].str.contains(r'featured image', case=False, regex=True).sum()
print(f"{featured_image_count} fake articles still contain 'featured image' after cleanup")

2062 fake articles still contain 'featured image' after cleanup


In [29]:
def remove_media_credits(text):
    # Catches "Featured Image:", "Featured image by...", "Featured image from..."
    text = re.sub(r'featured image[:\s](by|from)?.{0,60}', '', text, flags=re.IGNORECASE)
    return text

data['text'] = data['text'].apply(remove_media_credits)

fi_count = data['text'].str.contains('featured image', case=False, regex=True).sum()
print(f"{fi_count} articles still contain 'featured image'")

8 articles still contain 'featured image'


In [30]:
print("Reuters remaining:", data['text_clean'].str.contains('reuters').sum())
print("Via remaining:", data['text_clean'].str.contains(r'\bvia\b').sum())
print("Featured image remaining:", data['text'].str.contains('featured image', case=False).sum())

Reuters remaining: 20
Via remaining: 0
Featured image remaining: 8


In [31]:
print("Featured image in text_clean:", data['text_clean'].str.contains('featured image', case=False).sum())
print("Getty in text_clean:", data['text_clean'].str.contains('getty', case=False).sum())
print("Screenshot in text_clean:", data['text_clean'].str.contains('screenshot', case=False).sum())

Featured image in text_clean: 9
Getty in text_clean: 15
Screenshot in text_clean: 48


In [32]:
print("Getty in text_clean:", data['text_clean'].str.contains('getty', case=False).sum())
print("Screenshot in text_clean:", data['text_clean'].str.contains('screenshot', case=False).sum())

Getty in text_clean: 15
Screenshot in text_clean: 48


In [33]:
data.to_csv('../data/cleaned_news.csv', index=False)